# Dollar-Cost Averaging vs Lump-Sum Investing — France

Self-contained notebook: backtests LSI vs DCA on Euronext Paris tickers over rolling 5-year windows. No external project dependency — everything needed lives in the cells below.

Idle DCA cash earns the historical Livret A rate (not 0%), and prices are fetched raw (`auto_adjust=False`) so dividends aren't double-counted (once via price adjustment, once via the explicit DRIP loop) — the single biggest correctness issue found while building this.

## Setup

In [ ]:
!pip install -q pandas numpy scipy yfinance matplotlib seaborn python-dateutil openpyxl

In [ ]:
from __future__ import annotations
import numpy as np
import pandas as pd
import datetime as dt
import yfinance as yf
from dataclasses import dataclass
from scipy import stats
import logging
import time
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt
import seaborn as sns

import logging
import sys

## Risk-free rate for idle DCA cash — Livret A schedule

In [ ]:
SCHEDULE: list[tuple[str, float]] = [
    ("2009-08-01", 0.0125),
    ("2010-08-01", 0.0175),
    ("2011-02-01", 0.0200),
    ("2011-08-01", 0.0225),
    ("2013-02-01", 0.0175),
    ("2013-08-01", 0.0125),
    ("2014-08-01", 0.0100),
    ("2015-08-01", 0.0075),
    ("2020-02-01", 0.0050),
    ("2022-02-01", 0.0100),
    ("2022-08-01", 0.0200),
    ("2023-02-01", 0.0300),
]


_DATES = pd.to_datetime([d for d, _ in SCHEDULE])


_RATES = np.array([r for _, r in SCHEDULE])


def livret_a_rate(date: pd.Timestamp | str) -> float:
    """Annual Livret A rate in effect on `date`.

    Outside the known range, the nearest bound is extended flat (all study
    windows fall within 2009-2024).
    """
    idx = _DATES.searchsorted(pd.Timestamp(date), side="right") - 1
    return float(_RATES[np.clip(idx, 0, len(_RATES) - 1)])

## Data

In [ ]:
def fetch_price_history(ticker: str, t0: dt.datetime, tf: dt.datetime) -> pd.DataFrame:
    """OHLC + Dividends for `ticker` over [t0, tf], raw (auto_adjust=False).

    auto_adjust=False is mandatory here: yfinance's default (True) already
    folds dividends into the price series, and the simulation engine
    reinvests dividends explicitly via DRIP — combining both double-counts
    every dividend. Splits need no separate handling: yfinance always
    split-adjusts OHLC regardless of auto_adjust.
    """
    df = yf.Ticker(ticker).history(start=t0, end=tf + dt.timedelta(days=5), auto_adjust=False)
    if df.empty:
        raise ValueError(f"No data for {ticker} between {t0.date()} and {tf.date()}.")
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df = df.loc[t0:tf]
    if df.empty:
        raise ValueError(f"Empty segment for {ticker} between {t0.date()} and {tf.date()}.")
    return df

## Simulation engine — LSI and DCA

In [ ]:
FREQ_AZ = {"D": "B", "W": "W-MON", "M": "MS", "Q": "BQS", "S": "6MS", "A": "BYS"}


@dataclass(frozen=True)
class SimulationResult:
    strategy: str
    ticker: str
    t0: pd.Timestamp
    tf: pd.Timestamp
    w0: float
    w_final: float
    shares_final: float
    equity_curve: pd.Series

    @property
    def years(self) -> float:
        return (self.tf - self.t0).days / 365.25

    @property
    def hpr(self) -> float:
        return self.w_final / self.w0 - 1

    @property
    def cagr(self) -> float:
        return (self.w_final / self.w0) ** (1 / self.years) - 1 if self.years > 0 else float("nan")


def _lsi_from_prices(df: pd.DataFrame, w0: float) -> tuple[pd.Series, float]:
    """Buy w0 at t0's open, reinvest every dividend at that day's close."""
    shares = w0 / df.iloc[0]["Open"]
    equity = []
    for _, row in df.iterrows():
        if row["Dividends"] > 0 and shares > 0:
            shares += shares * row["Dividends"] / row["Close"]
        equity.append(shares * row["Close"])
    return pd.Series(equity, index=df.index), shares


def _tranche_dates(index: pd.DatetimeIndex, t0: dt.datetime, tf: dt.datetime, freq: str) -> list[pd.Timestamp]:
    """Map theoretical calendar dates (per `freq`) onto the next available session."""
    theoretical = pd.date_range(start=t0, end=tf, freq=FREQ_AZ[freq.upper()])
    mapped = set()
    for d in theoretical:
        available = index[index >= d]
        if not available.empty:
            mapped.add(available[0])
    return sorted(mapped)


def _dca_from_prices(
    df: pd.DataFrame, w0: float, dates: list[pd.Timestamp], cash_rf_annual: float | None,
) -> tuple[pd.Series, float, float]:
    """Invest w0/N at each tranche date's open; idle cash earns `cash_rf_annual`
    (or the Livret A schedule if None), compounded on calendar days elapsed."""
    tranche = w0 / len(dates)
    shares, cash = 0.0, w0
    prev_date = df.index[0]
    equity = []
    for date, row in df.iterrows():
        elapsed = (date - prev_date).days
        if elapsed > 0 and cash > 0:
            rate = cash_rf_annual if cash_rf_annual is not None else livret_a_rate(date)
            cash *= (1 + rate) ** (elapsed / 365)
        prev_date = date

        if date in dates:
            shares += tranche / row["Open"]
            cash -= tranche

        if row["Dividends"] > 0 and shares > 0:
            shares += shares * row["Dividends"] / row["Close"]

        equity.append(shares * row["Close"] + max(cash, 0.0))
    return pd.Series(equity, index=df.index), shares, cash


def simulate_lsi(ticker: str, w0: float, t0: dt.datetime, tf: dt.datetime) -> SimulationResult:
    df = fetch_price_history(ticker, t0, tf)
    equity, shares = _lsi_from_prices(df, w0)
    return SimulationResult(
        strategy="LSI", ticker=ticker, t0=df.index[0], tf=df.index[-1],
        w0=w0, w_final=float(equity.iloc[-1]), shares_final=shares, equity_curve=equity,
    )


def simulate_dca(
    ticker: str, w0: float, t0: dt.datetime, tf: dt.datetime,
    freq: str = "M", cash_rf_annual: float | None = None,
) -> SimulationResult:
    if freq.upper() not in FREQ_AZ:
        raise ValueError(f"Invalid frequency. Choose one of {list(FREQ_AZ)}.")
    df = fetch_price_history(ticker, t0, tf)
    dates = _tranche_dates(df.index, t0, tf, freq)
    if not dates:
        raise ValueError("Empty DCA schedule — check t0/tf/freq.")
    equity, shares, _cash = _dca_from_prices(df, w0, dates, cash_rf_annual)
    return SimulationResult(
        strategy="DCA", ticker=ticker, t0=df.index[0], tf=df.index[-1],
        w0=w0, w_final=float(equity.iloc[-1]), shares_final=shares, equity_curve=equity,
    )

## Metrics and statistical tests

In [ ]:
def sharpe_sortino(equity: pd.Series, rf_annual: float = 0.02) -> tuple[float, float]:
    """Annualized Sharpe/Sortino from a daily equity curve (log returns, 252 trading days)."""
    log_r = np.log(equity / equity.shift(1)).dropna()
    if len(log_r) < 5:
        return float("nan"), float("nan")

    excess = log_r - rf_annual / 252
    vol = excess.std(ddof=1)
    sharpe = excess.mean() / vol * np.sqrt(252) if vol > 0 else float("nan")

    downside = excess[excess < 0]
    down_std = downside.std(ddof=1) if len(downside) > 1 else float("nan")
    sortino = excess.mean() / down_std * np.sqrt(252) if down_std and down_std > 0 else float("nan")
    return sharpe, sortino


METRIC_COLUMNS = [
    ("HPR (%)", "lsi_hpr", "dca_hpr", 100),
    ("CAGR (%)", "lsi_cagr", "dca_cagr", 100),
    ("Sharpe", "lsi_sharpe", "dca_sharpe", 1),
    ("Sortino", "lsi_sortino", "dca_sortino", 1),
    ("Final wealth", "lsi_w_final", "dca_w_final", 1),
]


def summary_statistics(results: pd.DataFrame) -> pd.DataFrame:
    """Descriptive stats (mean/std/median/...) per metric, LSI vs DCA."""
    rows = []
    for label, lsi_col, dca_col, scale in METRIC_COLUMNS:
        lsi, dca = results[lsi_col].dropna() * scale, results[dca_col].dropna() * scale
        rows.append({
            "metric": label,
            "lsi_mean": lsi.mean(), "lsi_std": lsi.std(), "lsi_median": lsi.median(),
            "dca_mean": dca.mean(), "dca_std": dca.std(), "dca_median": dca.median(),
            "mean_gap": lsi.mean() - dca.mean(),
        })
    return pd.DataFrame(rows).set_index("metric")


def statistical_tests(results: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """Paired tests on HPR (LSI vs DCA share the same market path per window,
    so the paired design cancels common cross-sectional variance)."""
    paired = results[["lsi_hpr", "dca_hpr"]].dropna()
    diff = paired["lsi_hpr"] - paired["dca_hpr"]
    rows = []

    stat, p = stats.mannwhitneyu(paired["lsi_hpr"], paired["dca_hpr"], alternative="two-sided")
    _, p_greater = stats.mannwhitneyu(paired["lsi_hpr"], paired["dca_hpr"], alternative="greater")
    rows.append({
        "test": "Mann-Whitney U", "statistic": stat, "p_value": p,
        "significant": p < alpha,
        "conclusion": ("LSI > DCA" if p_greater < alpha else "DCA > LSI") if p < alpha else "no significant difference",
    })

    if len(diff) >= 10 and not np.all(diff == 0):
        stat, p = stats.wilcoxon(diff, alternative="two-sided")
        median = diff.median()
        rows.append({
            "test": "Wilcoxon signed-rank (paired)", "statistic": stat, "p_value": p,
            "significant": p < alpha,
            "conclusion": (f"LSI > DCA (median Δ={median*100:+.2f}%)" if median > 0
                           else f"DCA > LSI (median Δ={median*100:+.2f}%)") if p < alpha else "no significant difference",
        })

    return pd.DataFrame(rows).set_index("test")


def regime_breakdown(results: pd.DataFrame) -> pd.DataFrame:
    """LSI vs DCA outcomes split by entry-market tertile (low/mid/high)."""
    df = results.copy()
    df["regime"] = pd.qcut(df["entry_level"], q=3, labels=["Low entry", "Mid entry", "High entry"])
    rows = []
    for regime, group in df.groupby("regime", observed=True):
        rows.append({
            "regime": regime, "n_windows": len(group),
            "lsi_hpr_mean": group["lsi_hpr"].mean() * 100,
            "dca_hpr_mean": group["dca_hpr"].mean() * 100,
            "median_gap": (group["lsi_hpr"] - group["dca_hpr"]).median() * 100,
            "lsi_win_rate": (group["hpr_diff"] > 0).mean() * 100,
        })
    return pd.DataFrame(rows).set_index("regime")

## Rolling-window orchestration

In [ ]:
logger = logging.getLogger("lsi_dca")


def run_rolling_windows(
    ticker: str, w0: float, t0: str, *, window_years: int, shift_months: int,
    n_windows: int, dca_freq: str = "M", rf_annual: float = 0.02,
    cash_rf_annual: float | None = None,
) -> pd.DataFrame:
    """Simulate LSI vs DCA over `n_windows` overlapping windows.

    Each window i spans [t0 + i*shift_months, t0 + i*shift_months + window_years].
    Returns one row per window: HPR/CAGR/Sharpe/Sortino for both strategies,
    plus the entry price level used for regime tagging downstream.
    """
    t0_base = pd.Timestamp(t0)
    records = []
    started = time.perf_counter()

    for i in range(n_windows):
        w_t0 = t0_base + relativedelta(months=i * shift_months)
        w_tf = w_t0 + relativedelta(years=window_years)
        try:
            lsi = simulate_lsi(ticker, w0, w_t0, w_tf)
            dca = simulate_dca(ticker, w0, w_t0, w_tf, freq=dca_freq, cash_rf_annual=cash_rf_annual)
        except Exception as exc:  # noqa: BLE001 — a bad window shouldn't kill the whole sweep
            logger.debug("Window %d/%d skipped (%s): %s", i + 1, n_windows, ticker, exc)
            continue

        lsi_sharpe, lsi_sortino = sharpe_sortino(lsi.equity_curve, rf_annual)
        dca_sharpe, dca_sortino = sharpe_sortino(dca.equity_curve, rf_annual)
        records.append({
            "window": i + 1, "t0": lsi.t0, "tf": lsi.tf, "entry_level": lsi.equity_curve.iloc[0],
            "lsi_hpr": lsi.hpr, "lsi_cagr": lsi.cagr, "lsi_sharpe": lsi_sharpe, "lsi_sortino": lsi_sortino,
            "lsi_w_final": lsi.w_final,
            "dca_hpr": dca.hpr, "dca_cagr": dca.cagr, "dca_sharpe": dca_sharpe, "dca_sortino": dca_sortino,
            "dca_w_final": dca.w_final,
            "hpr_diff": lsi.hpr - dca.hpr,
        })
        logger.debug(
            "[%s] window %d/%d %s->%s | LSI HPR=%.1f%% DCA HPR=%.1f%%",
            ticker, i + 1, n_windows, w_t0.date(), w_tf.date(), lsi.hpr * 100, dca.hpr * 100,
        )

    if not records:
        raise RuntimeError(f"No window could be simulated for {ticker} — check parameters.")

    elapsed = time.perf_counter() - started
    logger.info("%-10s | %d/%d windows | DCA %s | %.1fs", ticker, len(records), n_windows, dca_freq, elapsed)
    return pd.DataFrame(records)


@dataclass
class BacktestReport:
    ticker: str
    results: pd.DataFrame
    summary: pd.DataFrame
    tests: pd.DataFrame
    regimes: pd.DataFrame

    @property
    def lsi_win_rate(self) -> float:
        return (self.results["hpr_diff"] > 0).mean() * 100


def compare(
    ticker: str, w0: float, t0: str, *, window_years: int = 5, shift_months: int = 1,
    n_windows: int = 60, dca_freq: str = "M", rf_annual: float = 0.02,
    cash_rf_annual: float | None = None, alpha: float = 0.05,
) -> BacktestReport:
    """Full LSI-vs-DCA comparison for one ticker/frequency: rolling windows +
    descriptive stats + paired significance tests + regime breakdown."""
    results = run_rolling_windows(
        ticker, w0, t0, window_years=window_years, shift_months=shift_months,
        n_windows=n_windows, dca_freq=dca_freq, rf_annual=rf_annual, cash_rf_annual=cash_rf_annual,
    )
    return BacktestReport(
        ticker=ticker,
        results=results,
        summary=summary_statistics(results),
        tests=statistical_tests(results, alpha=alpha),
        regimes=regime_breakdown(results),
    )


def backtest_frequencies(
    ticker: str, w0: float = 12_000, t0: str = "2010-01-04", *,
    freqs: tuple[str, ...] = ("M", "S", "A"), **kwargs,
) -> pd.DataFrame:
    """Compare() at several DCA frequencies, stacked into one tidy table
    (mirrors the paper's Table 1 layout: metric x frequency)."""
    rows = []
    for freq in freqs:
        report = compare(ticker, w0, t0, dca_freq=freq, **kwargs)
        for label, lsi_col, dca_col, scale in METRIC_COLUMNS:
            lsi = report.results[lsi_col].dropna() * scale
            dca = report.results[dca_col].dropna() * scale
            rows.append({
                "dca_frequency": freq, "metric": label,
                "lsi_mean": lsi.mean(), "lsi_std": lsi.std(),
                "dca_mean": dca.mean(), "dca_std": dca.std(),
                "mean_gap": lsi.mean() - dca.mean(),
                "lsi_win_rate_pct": (report.results[lsi_col] > report.results[dca_col]).mean() * 100,
            })
    return pd.DataFrame(rows).set_index(["dca_frequency", "metric"])

## Charts

In [ ]:
CLR_LSI, CLR_DCA, CLR_DIFF = "#E63946", "#457B9D", "#2A9D8F"


STYLE = "seaborn-v0_8-whitegrid"


def equity_curves(lsi_curve: pd.Series, dca_curve: pd.Series, *, title: str = "") -> plt.Figure:
    """Overlay LSI vs DCA wealth path for a single window."""
    with plt.style.context(STYLE):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(lsi_curve.index, lsi_curve.values, color=CLR_LSI, lw=1.8, label="Lump-Sum")
        ax.plot(dca_curve.index, dca_curve.values, color=CLR_DCA, lw=1.8, label="DCA")
        ax.set_title(title or "Portfolio value over time")
        ax.set_ylabel("Portfolio value")
        ax.legend(frameon=False)
        fig.tight_layout()
    return fig


def hpr_distribution(results: pd.DataFrame, *, ticker: str = "") -> plt.Figure:
    """Distribution of Holding-Period Return across all rolling windows."""
    long = pd.concat([
        results[["lsi_hpr"]].rename(columns={"lsi_hpr": "hpr"}).assign(strategy="Lump-Sum"),
        results[["dca_hpr"]].rename(columns={"dca_hpr": "hpr"}).assign(strategy="DCA"),
    ])
    long["hpr"] *= 100
    with plt.style.context(STYLE):
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.violinplot(data=long, x="strategy", y="hpr", hue="strategy",
                        palette={"Lump-Sum": CLR_LSI, "DCA": CLR_DCA}, legend=False, ax=ax)
        ax.set_title(f"HPR distribution across rolling windows — {ticker}".strip(" —"))
        ax.set_ylabel("Holding-period return (%)")
        ax.set_xlabel("")
        fig.tight_layout()
    return fig


def hpr_gap_timeline(results: pd.DataFrame, *, ticker: str = "") -> plt.Figure:
    """LSI - DCA gap in HPR against the window's entry date."""
    with plt.style.context(STYLE):
        fig, ax = plt.subplots(figsize=(9, 5))
        gap = results["hpr_diff"] * 100
        colors = [CLR_LSI if v > 0 else CLR_DCA for v in gap]
        ax.bar(results["t0"], gap, color=colors, width=20)
        ax.axhline(0, color="black", lw=0.8)
        ax.set_title(f"LSI − DCA return gap by entry date — {ticker}".strip(" —"))
        ax.set_ylabel("HPR gap (percentage points)")
        fig.tight_layout()
    return fig


def win_rate_by_regime(regimes: pd.DataFrame, *, ticker: str = "") -> plt.Figure:
    """LSI win rate split by entry-market tertile (low / mid / high)."""
    with plt.style.context(STYLE):
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.bar(regimes.index.astype(str), regimes["lsi_win_rate"], color=CLR_DIFF)
        ax.axhline(50, color="black", lw=0.8, ls="--")
        ax.set_ylim(0, 100)
        ax.set_ylabel("LSI win rate (%)")
        ax.set_title(f"LSI win rate by entry regime — {ticker}".strip(" —"))
        fig.tight_layout()
    return fig


def risk_return(results: pd.DataFrame, *, ticker: str = "") -> plt.Figure:
    """Mean Sharpe vs mean HPR — LSI vs DCA, one point each."""
    with plt.style.context(STYLE):
        fig, ax = plt.subplots(figsize=(6, 6))
        for label, hpr_col, sharpe_col, color in [
            ("Lump-Sum", "lsi_hpr", "lsi_sharpe", CLR_LSI),
            ("DCA", "dca_hpr", "dca_sharpe", CLR_DCA),
        ]:
            ax.scatter(results[sharpe_col].mean(), results[hpr_col].mean() * 100,
                       s=160, color=color, label=label, zorder=3)
        ax.set_xlabel("Mean Sharpe ratio")
        ax.set_ylabel("Mean HPR (%)")
        ax.set_title(f"Risk vs return — {ticker}".strip(" —"))
        ax.legend(frameon=False)
        fig.tight_layout()
    return fig

## Logging — INFO shows one line per run, DEBUG shows one line per window

In [ ]:
def configure_logging(level=logging.INFO):
    handler = logging.StreamHandler(stream=sys.stdout)
    handler.setFormatter(logging.Formatter("%(message)s"))
    logger = logging.getLogger("lsi_dca")
    logger.handlers.clear()
    logger.addHandler(handler)
    logger.setLevel(level)
    logger.propagate = False

configure_logging(logging.INFO)
%matplotlib inline

## Table 1 — CAC 40 index, three DCA frequencies

$W_0 = 12{,}000$€, $t_0=$ 2010-01-04, 5-year rolling windows, 60 windows, 1-month shift.

In [ ]:
COMMON = dict(w0=12_000, t0="2010-01-04", window_years=5, shift_months=1, n_windows=60, rf_annual=0.02)

table1 = backtest_frequencies("^FCHI", **COMMON, freqs=("M", "S", "A"))
table1

## A closer look — one representative window (CAC 40, semi-annual DCA)

In [ ]:
report = compare("^FCHI", **COMMON, dca_freq="S")
report.summary

In [ ]:
first_t0, first_tf = report.results['t0'].iloc[0], report.results['tf'].iloc[0]
lsi = simulate_lsi("^FCHI", 12_000, first_t0, first_tf)
dca = simulate_dca("^FCHI", 12_000, first_t0, first_tf, freq="S")

equity_curves(lsi.equity_curve, dca.equity_curve, title="CAC 40 — portfolio value, first window")

In [ ]:
hpr_distribution(report.results, ticker="CAC 40")

In [ ]:
hpr_gap_timeline(report.results, ticker="CAC 40")

In [ ]:
win_rate_by_regime(report.regimes, ticker="CAC 40")

In [ ]:
risk_return(report.results, ticker="CAC 40")

## Statistical significance

In [ ]:
report.tests

## Table 2 — Cross-section, 38 CAC 40 constituents (semi-annual DCA)

In [ ]:
CONSTITUENTS = {
    "Air Liquide": "AI.PA", "Airbus": "AIR.PA", "AXA": "CS.PA", "BNP Paribas": "BNP.PA",
    "Bouygues": "EN.PA", "Capgemini": "CAP.PA", "Carrefour": "CA.PA", "Credit Agricole": "ACA.PA",
    "Danone": "BN.PA", "Dassault Systemes": "DSY.PA", "Engie": "ENGI.PA", "EssilorLuxottica": "EL.PA",
    "Eurofins Scientific": "ERF.PA", "Hermes": "RMS.PA", "Kering": "KER.PA", "Legrand": "LR.PA",
    "L'Oreal": "OR.PA", "LVMH": "MC.PA", "Michelin": "ML.PA", "Orange": "ORA.PA",
    "Pernod Ricard": "RI.PA", "Publicis Groupe": "PUB.PA", "Renault": "RNO.PA", "Safran": "SAF.PA",
    "Saint-Gobain": "SGO.PA", "Sanofi": "SAN.PA", "Schneider Electric": "SU.PA",
    "Societe Generale": "GLE.PA", "Stellantis": "STLAM.MI", "Alstom": "ALO.PA",
    "Teleperformance": "TEP.PA", "Thales": "HO.PA", "TotalEnergies": "TTE.PA", "Veolia": "VIE.PA",
    "Vinci": "DG.PA", "Vivendi": "VIV.PA", "Accor": "AC.PA", "Christian Dior": "CDI.PA",
}

rows = []
for name, ticker in CONSTITUENTS.items():
    try:
        r = compare(ticker, **COMMON, dca_freq="S")
        rows.append({
            "Company": name, "Ticker": ticker,
            "LSI HPR mean (%)": r.summary.loc["HPR (%)", "lsi_mean"],
            "LSI HPR std (%)": r.summary.loc["HPR (%)", "lsi_std"],
            "DCA HPR mean (%)": r.summary.loc["HPR (%)", "dca_mean"],
            "DCA HPR std (%)": r.summary.loc["HPR (%)", "dca_std"],
            "LSI win rate (%)": r.lsi_win_rate,
        })
    except Exception as e:
        print(f"skipped {name} ({ticker}): {e}")

table2 = pd.DataFrame(rows).set_index("Company").sort_values("LSI win rate (%)")
table2

## Robustness — idle DCA cash at 0% vs the Livret A schedule

Re-runs the cross-section with `cash_rf_annual=0.0` (the original zero-yield assumption) to quantify how much of DCA's shortfall was simply un-remunerated cash.

In [ ]:
robust_rows = []
for name, ticker in CONSTITUENTS.items():
    try:
        no_yield = compare(ticker, **COMMON, dca_freq="S", cash_rf_annual=0.0)
        with_yield = compare(ticker, **COMMON, dca_freq="S", cash_rf_annual=None)
        robust_rows.append({
            "Company": name, "Ticker": ticker,
            "DCA mean HPR before (%)": no_yield.summary.loc["HPR (%)", "dca_mean"],
            "DCA mean HPR after (%)": with_yield.summary.loc["HPR (%)", "dca_mean"],
            "LSI win rate before (%)": no_yield.lsi_win_rate,
            "LSI win rate after (%)": with_yield.lsi_win_rate,
            "Win rate change (pts)": with_yield.lsi_win_rate - no_yield.lsi_win_rate,
        })
    except Exception as e:
        print(f"skipped {name} ({ticker}): {e}")

robustness = pd.DataFrame(robust_rows).set_index("Company").sort_values("Win rate change (pts)")
robustness

## Export — consolidated Excel workbook

In [ ]:
with pd.ExcelWriter("dca_lsi_results.xlsx", engine="openpyxl") as writer:
    table1.to_excel(writer, sheet_name="Table 1 - CAC40 Index")
    table2.to_excel(writer, sheet_name="Table 2 - Cross Section")
    robustness.to_excel(writer, sheet_name="Cash Yield Robustness")
print("Exported -> dca_lsi_results.xlsx")